# Lentiscreen Analysis 

The script was last updated: 20-07-2024
Written by: Estelle Deby (estelle.deby@epfl.ch)
Affiliation: Ecole Polytechnique de Lausanne (EPFL) Switzerland 

<b><u> Description </b></u>

This script is used to process tabular data output from the QuPath interface and generate a comprehensive overview of the data at hand. Ultimatelty, we aim to identify transcription factors that may phenocopy Arid1a loss. In the following code the terms 'CRE', 'Arid1a-/-' are used interchangebly to define 'Arid1a KO' organoids. Similarly, the terms 'WT' or 'Arid1a fl/fl' refer to 'Arid1a WT' organoids.

In [ ]:
#import packages
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

from matplotlib import rcParams
rcParams['ps.fonttype'] = 42
rcParams['pdf.fonttype'] = 42

#Set plotting styles
sns.set_theme('talk')
sns.set_style('whitegrid')


In [ ]:
#Load the datasets 
df1 = pd.read_csv('/Users/deby/Desktop/lentiscreen_data/2_Image_analysis/2.3 - QuPath_output/LentriScreen_annotations.csv', comment ='#')
#display(df1)
df2 = pd.read_csv('/Users/deby/Desktop/lentiscreen_data/2_Image_analysis/2.3 - QuPath_output/LentriScreen_detections.csv', comment ='#')
#display(df2)

In [ ]:
# Defining a folder path for the graphics 
folder_path = "/Users/deby/Desktop/lentiscreen_data/4_Data/4.2_Graphs"
# Ensure the folder exists
os.makedirs(folder_path, exist_ok=True)

# Create a new path for the statistics outout
folder_path_stats = '/Users/deby/Desktop/lentiscreen_data/4_Data/4.3_Results_statistics'
os.makedirs(folder_path_stats, exist_ok=True)

## 0. Check datasets for duplicates

Sanity check to ensure that we do not have duplicates in the dataset

In [ ]:
# Removing duplicate rows based on 'Centroid X µm' and 'Centroid Y µm'
df1 = df1.drop_duplicates(subset=['Centroid X µm', 'Centroid Y µm'])
print("Original DataFrame:")
display(df1)
print("DataFrame after removing duplicate rows based on 'Centroid X µm' and 'Centroid Y µm':")
display(df1)

## 1. Create an Organoid-tag in the first dataframe and calculate the mean thickness of the ridge foreach cystic organoid in each condition 

First, the dataframe is formatted, 1. to add important information included in the image name to the dataframe, 2. based on this information add a 'tag' to each row. This approach will be repeated in step 2. for the second dataframe, with a final goal to merge both dataframes on the created 'tag'.

In [ ]:

## Extract information from the image name ('Image' column)
def extract_org_id(image_col): #extract the number/ID of the genetically modified organoid line
    match = re.search(r'(\S+)\s*-\s*\d+$', image_col)
    if match:
        return match.group(1)
    return np.nan  # Return NaN if no match is found

def extract_date(image_col): #extract the date of the experiment
    match = re.search('^(\d{8})', image_col)
    if match:
        return match.group(1)
    return np.nan  # Return NaN if no match is found

def extract_mmo1type(image_col): #extract the mmo1_line 
    match = re.search(r'mmo1\s+(\w{2,3})', image_col)
    if match: 
        return match.group(1)
    return np.nan # Return NaN if no match is found

def extract_frame_id(image_col):
    match = re.search(r'(\d{1,2})\D*$', image_col)
    if match:
        return match.group(1)
    return np.nan  # Return NaN if no match is found

# Add the extracted information from the image names to new columns in the dataframe 
df2["GEMorganoid"] = df2["Image"].apply(extract_org_id)
df2["date"] = df2["Image"].apply(extract_date)
df2['mmo1_line'] = df2 ["Image"].apply(extract_mmo1type)
df2["image_frame"] =df2["Image"].apply(extract_frame_id)
display(df2)

#controls have been labeled using different terminology on the Leica Microscope // here I uniform the naming
values_to_replace = {'Image', 'wt', 'ko', 'Series', 'channels', 'P', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'Parental'}
condition = df2['GEMorganoid'].str.contains('|'.join(values_to_replace)) 
df2.loc[condition, 'GEMorganoid'] = 'controls' # Replace values in 'organoid_id' column with 'controls' based on the condition
values_to_replace2 = {'Ctrl'}
condition2 = df2['GEMorganoid'].str.contains('|'.join(values_to_replace2)) # Create a condition using the | (OR) operator for each value in the set
df2.loc[condition2, 'GEMorganoid'] = 'Negative_control' # Replace values in 'organoid_id' column with 'Negative_control' based on the condition
display(df2) # Display the updated DataFrame
print(df2.columns)
df2.columns = df2.columns.str.strip()  # Remove leading and trailing whitespaces

df_thickness = df2[['Image', 'Classification', 'Thickness µm', 'Organoid ID', 'GEMorganoid', 'date', 'mmo1_line', 'image_frame']]
display(df_thickness)

#remove all failed measuremnts that have a thickness measurement of 0, since the QuPath software will label a failed measurements as a '0' measurement 
df_thickness_filtered = df_thickness[df_thickness['Thickness µm'] > 0]
display(df_thickness_filtered)

# REPLACE GUIDE NUMBER BY GUIDE NAME AND NUMBER 
def transform_guide_to_name(s):
    name_mappings2 = {
        '42': 'Foxa1_sgRNA1',
        '43': 'Foxa1_sgRNA2',
        '44': 'Foxa1_sgRNA3',
        '50': 'Meis1_sgRNA3',
        '49': 'Meis1_sgRNA2',
        '48': 'Meis1_sgRNA1',
        '51': 'Meis2_sgRNA1',
        '52': 'Meis2_sgRNA2',
        '53': 'Meis2_sgRNA3',
        '69': 'Sox9_sgRNA1',
        '70': 'Sox9_sgRNA2',
        '71': 'Sox9_sgRNA3',
        '75': 'Tcf12_sgRNA3',
        '76': 'Tcf12_sgRNA2',
        '77': 'Tcf12_sgRNA1',
        '33': 'Atf2_sgRNA1',
        '34': 'Atf2_sgRNA2',
        '35': 'Atf2_sgRNA3',
        '81': 'Zpf467_sgRNA1',
        '82': 'Zpf467_sgRNA2',
        '83': 'Zpf467_sgRNA3',
        '84': 'Esr1_sgRNA1',
        '85': 'Esr1_sgRNA2',
        '86': 'Esr1_sgRNA3',
        '87': 'Gata3_sgRNA1',
        '88': 'Gata3_sgRNA2',
        '89': 'Gata3_sgRNA3',
        '57': 'Nfib_sgRNA1',
        '58': 'Nfib_sgRNA2',
        '104': 'cFOS_sgRNA1',
        '105': 'cFOS_sgRNA2',
        '30': 'Ap2a_sgRNA1',
        '31': 'Ap2a_sgRNA2',
        '32': 'Ap2a_sgRNA3',
        '36': 'Atf3_sgRNA1',
        '37': 'Atf3_sgRNA2',
        '38': 'Atf3_sgRNA3',
        '39': 'Elf5_sgRNA1',
        '40': 'Elf5_sgRNA2',
        '41': 'Elf5_sgRNA3',
        '45': 'Irf1_sgRNA1',
        '46': 'Irf1_sgRNA2',
        '47': 'Irf1_sgRNA3',
        '49': 'Meis1_sgRNA2',
        '48': 'Meis1_sgRNA1',
        '51': 'Meis2_sgRNA1',
        '52': 'Meis2_sgRNA2',
        '53': 'Meis2_sgRNA3',
        '54': 'Nfatc1_sgRNA1',
        '55': 'Nfatc1_sgRNA2',
        '56': 'Nfatc1_sgRNA3',
        '60': 'Relb_sgRNA1',
        '61': 'Relb_sgRNA2',
        '62': 'Relb_sgRNA3',
        '63': 'Runx1_sgRNA1',
        '64': 'Runx1_sgRNA2',
        '65': 'Runx1_sgRNA3',
        '66': 'Runx2_sgRNA1',
        '67': 'Runx2_sgRNA2',
        '68': 'Runx2_sgRNA3',
        '69': 'Sox9_sgRNA1',
        '70': 'Sox9_sgRNA2',
        '71': 'Sox9_sgRNA3',
        '72': 'Stat1_sgRNA1',
        '73': 'Stat1_sgRNA2',
        '74': 'Stat1_sgRNA3',
        '75': 'Tcf12_sgRNA3',
        '76': 'Tcf12_sgRNA2',
        '77': 'Tcf12_sgRNA1',
        '78': 'Tead1_sgRNA1',
        '79': 'Tead1_sgRNA2',
        '80': 'Tead1_sgRNA3',
        '33': 'Atf2_sgRNA1',
        '34': 'Atf2_sgRNA2',
        '35': 'Atf2_sgRNA3',
        '81': 'Zpf467_sgRNA1',
        '82': 'Zpf467_sgRNA2',
        '83': 'Zpf467_sgRNA3',
        '84': 'Esr1_sgRNA1',
        '85': 'Esr1_sgRNA2',
        '86': 'Esr1_sgRNA3',
        '87': 'Gata3_sgRNA1',
        '88': 'Gata3_sgRNA2',
        '89': 'Gata3_sgRNA3',
        '57': 'Nfib_sgRNA1',
        '58': 'Nfib_sgRNA2',
        '90': 'NT_sgRNA1',
        '91': 'NT_sgRNA2',
        'A1': 'Arid1a_sgRNA1',
        'A2': 'Arid1a_sgRNA2',
        'A3': 'Arid1a_sgRNA3', 
        'a1': 'Arid1a_sgRNA1',
        'P': 'Parental',
        'P3': 'Parental',
        'P4': 'Parental',
        'Neg': 'Negative_control',
        'neg1': 'Negative_control',
        'neg2': 'Negative_control',
        '2': 'Negative_control',
        'P': 'Parental',
        'P5': 'Parental',
        'P6': 'Parental',
        'P7': 'Parental',
        'P8': 'Parental',
        'P9': 'Parental',
        'P10': 'Parental',
        'Parental': 'Parental'
    }
    
    return name_mappings2.get(s, s)

# Apply the function to the 'organoid_id' column
df_thickness_filtered['GEMorganoid'] = df_thickness_filtered['GEMorganoid'].apply(transform_guide_to_name)
display(df_thickness_filtered)

df_thickness_filtered['date_sgRNA'] = df_thickness_filtered['date'].astype(str) + "_" + df_thickness_filtered['GEMorganoid'].astype(str)

##ensure we give the cystic organoids a tag so we can map them back to the second dataframe later 
df_thickness_filtered['Organoid_tag'] = df_thickness_filtered['date'].astype(str) + "_" + df_thickness_filtered['mmo1_line'].astype(str) + "_" + df_thickness_filtered['GEMorganoid'].astype(str) + "_" + df_thickness_filtered['Organoid ID'].astype(str) + "_" + df_thickness_filtered['image_frame'].astype(str)
display(df_thickness_filtered)

df_thickness_filtered.to_csv('df_thickness_filtered.csv', index=False)

For each individual organoid, we compute the mean thickness of the ridge as well as the standard deviation of the mean thickness. These are stored in new columns.

In [ ]:
# Group by 'date_sgRNA', 'Organoid_ID', and 'mmo1_line' and calculate mean and standard deviation
thickness_measurements = df_thickness_filtered.groupby(['date_sgRNA', 'GEMorganoid', 'Organoid ID', 'image_frame', 'Organoid_tag', 'mmo1_line'], as_index=False)['Thickness µm'].agg(['mean', 'std']).reset_index() 

# Display and save the new DataFrame with mean and standard deviation values
display(thickness_measurements)
thickness_measurements.to_csv('thickness_measurements.csv', index=False)

## 2. Create the Organoid-tag in the second dataframe 

In [ ]:
## Similarly as in point 1, extract information from the image name and add it to the dataframe 
df1["organoid_id"] = df1["Image"].apply(extract_org_id)
df1["date"] = df1["Image"].apply(extract_date)
df1['mmo1_line'] = df1 ["Image"].apply(extract_mmo1type)
df1["image_frame"] =df1["Image"].apply(extract_frame_id)

# Define organoid types
df3 = df1[df1['Classification'].isin(['glandular', 'cystic', 'intermediate', 'undefined', 'undefinable', 'Empty'])]
display(df3)
# Uniform the labelling of control conditions, similarly as above the naming was not done uniformely during image aquisition. This is accounted for in the following line.
values_to_replace = {'Image', 'wt', 'ko', 'Series', 'channels', 'P', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'Parental'}
condition = df3['organoid_id'].str.contains('|'.join(values_to_replace)) # Create a condition using the | (OR) operator for each value in the set
df3.loc[condition, 'organoid_id'] = 'controls' # Replace values in 'organoid_id' column with 'Ctrl' based on the condition
display(df3)

# Apply the function to the 'organoid_id' column
df3['organoid_id'] = df3['organoid_id'].apply(transform_guide_to_name)

##ensure we give the cystic organoids a 'tag' so we can map them back to the second frame later 
df3['Organoid_tag'] = df3['date'].astype(str) + "_" + df3['mmo1_line'].astype(str) + "_" + df3['organoid_id'].astype(str) + "_" + df3['Organoid ID'].astype(str) + "_" + df3['image_frame'].astype(str)
display(df3)


## 3. Merging of the dataframes based on the Organoid-tag 

In [ ]:
#Merge the two dataframes to have the measurements of the ridge thickness in the dataframe containing phenotype spread information
new_df = pd.merge(df3 , thickness_measurements, how='left', on='Organoid_tag')
display(new_df)
new_df.to_csv('new_df.csv', index=False)

## 4. Processing the merged data


### 4.A. Check data distribution, and thickness parameters.

In order to perform the right statistical tests, we check for the distribution of the data. Shapiro-Wilk test for normality is chosen and performed to check for the distribution of the ridge thickness variables.

In [ ]:
from scipy.stats import shapiro 

In [ ]:
## Shapiro thickness distribution
metric = 'mean'
values_to_test = new_df[new_df['Classification'].isin(['cystic', 'single-layered, cystic'])][metric].dropna()

#Set plotting styles
sns.set_style('ticks')

# plot the mean ridge thickness distribution 
ax = values_to_test.plot.kde(figsize=(8,8))
_, shapiro_pval = shapiro(values_to_test)
ax.set_title(f'Ridge thickness distribution, Shapiro p-value,: {"{:.2E}".format(shapiro_pval)}',  fontsize= 12)
ax.set_xlabel(metric)
ax.set_xlabel('Mean ridge thickness')
sns.despine()
plt.tight_layout()
#plt.savefig(folder_path +  "/shapiro_meanridgethickness.pdf", dpi=300)

In [ ]:
## Shapiro STD of the mean thickness /
metric = 'std'
values_to_test = new_df[new_df['Classification'].isin(['cystic', 'single-layered, cystic'])][metric].dropna()
#Set plotting styles
sns.set_style('ticks')

# plot
ax = values_to_test.plot.kde(figsize=(8,8))
_, shapiro_pval = shapiro(values_to_test)
ax.set_title(f'Standard deviation of the mean Ridge Thickness distribution, Shapiro p-value: {"{:.2E}".format(shapiro_pval)}', fontsize= 12)
ax.set_xlabel(metric)
ax.set_xlabel('Standard deviation of the mean ridge thickness')
sns.despine()
plt.tight_layout()
#plt.savefig(folder_path +  "/shapiro_STDmeanridgethickness.pdf", dpi=300)

Conlcusion - 
We performed a Shapiro-Wilk test for normality.The p-values are 2.21E-30 for the variable 'mean of the organoid ridge thickness' and 3.88E-42 of the variable 'SD of the mean of the organoid ridge thickness'. Therefore, these refute the null hypothesis that the data was drawn from a normal distribution. The downstream statistical tests were chosen to suit the non-normal distributions.

In the following part, we assess if the thickness of the organoid ridge and its standard deviation depend on the genetic status of Arid1a.
First the analysis consists in visualising differences, and then performing statistical analyses to define potential differences.

In [ ]:
## Testing on parental Arid1a fl/fl vs. Arid1a-/- organoids
thickness_measurements_ctrls = new_df[(new_df['Classification'] == 'cystic') & (new_df.date_sgRNA.str.endswith('controls'))]
sns.set_style('ticks')

# Map original labels to desired labels
legend_labels = {"WT": "Arid1a fl/fl", "CRE": "Arid1a-/-"}
# Show the joint distribution using kernel density estimation
g = sns.jointplot(
    data=thickness_measurements_ctrls,
    x="mean", y="std", hue="mmo1_line_x",
    palette='Set2',
    height=6,  
)    
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
plt.suptitle('Joint distribution for parental organoids', fontsize=14)

# Update legend labels
for t, l in zip(g.ax_joint.legend_.texts, legend_labels.values()):
    t.set_text(l)
g.ax_joint.legend_.set_title('Genetic Background') 
# Set y-axis ticks
g.ax_joint.set_yticks([10, 20, 30])
plt.tight_layout()
#g.fig.savefig(folder_path + '/joint_distribution_parental_organoids.pdf', dpi=300)
plt.show()

Next, we use the two-sample Kolmogorov-Smirnov test for goodness of fit to check whether the distributions of 1. the mean ridge thickness, and 2. standard deviation of the mean ridge thickness are different, respectively, for parental Arid1a fl/fl vs Arid1a -/- organoids.

In [ ]:
from scipy.stats import ks_2samp
print (ks_2samp(
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'WT']['mean'], 
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'CRE']['mean']
))

In [ ]:
print (ks_2samp(
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'WT']['std'], 
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'CRE']['std']
))

Conclusion: For both tests performed, the p-values are less than the significance level. We thereby reject the null hypotheses, concluding that there is a significant differences between the two distributions. 

In [ ]:
### For all organoids, we visualize mean ridge thickness and stratify based on Arid1a genetic status.
sns.set_theme(style="whitegrid", palette="muted")
# Define legend labels
y_labels = {"WT": "Arid1a fl/fl", "CRE": "Arid1a-/-"}

# Draw a categorical scatterplot to show each observation
plt.figure(figsize=(10,5))
plt.title('Mean ridge thickness of cystic, parental organoids', fontsize=14)
plt.xlabel('Mean Thickness (uM)', fontsize=14, labelpad=10)

ax = sns.swarmplot(data=thickness_measurements_ctrls, x="mean", y="mmo1_line_x", hue="std")
ax.set(ylabel="")

# Set y-axis labels
ax.set_yticklabels([y_labels[label.get_text()] for label in ax.get_yticklabels()])
plt.tight_layout()
#plt.savefig(folder_path + '/mean_ridgethickness_allparental_cysticorganoids.pdf', dpi=300)
plt.show()

We perform additional controls and checks, to to check whether the distributions of 1. the mean ridge thickness, and 2. the standard deviation of the mean ridge thickness are different, respectively, for parental Arid1a fl/fl vs transduced Arid1a fl/fl with non-targeting sgRNA.The analysis is repeated for for parental Arid1a -/- vs. transduced Arid1a -/- with non-targeting sgRNAs.

In [ ]:
#0. Curate the data to later plot it.
thickness_screen_checks = thickness_measurements[thickness_measurements.date_sgRNA.str.endswith('controls') + thickness_measurements.GEMorganoid.str.startswith('NT_sgRNA')]
#thickness_screen_checks = new_df[new_df.date_sgRNA.str.endswith('controls') + new_df.GEMorganoid.str.startswith('NT_sgRNA')]
thickness_screen_checks_wt = thickness_screen_checks[thickness_screen_checks['mmo1_line'] == 'WT']
thickness_screen_checks_cre = thickness_screen_checks[thickness_screen_checks['mmo1_line'] == 'CRE']


## 1. Parental Arid1a fl/fl vs transduced Arid1a fl/fl organoids with non-targeting sgRNAs
sns.set_style('ticks')
legend_labels = {"WT": "Arid1a fl/fl", "CRE": "Arid1a-/-"} # Map original labels to desired labels
#Show the joint distribution using kernel density estimation
order = ['controls', 'NT_sgRNA1', 'NT_sgRNA2'] # Define the desired order of categories
thickness_screen_checks_wt['GEMorganoid'] = pd.Categorical(thickness_screen_checks_wt['GEMorganoid'], categories=order, ordered=True) # Reorder the levels of the "GEMorganoid" variable
g = sns.jointplot(
    data=thickness_screen_checks_wt,
    x="mean", y="std", hue="GEMorganoid",
    #kind="kde",
    palette='Set2',  
)
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
g.ax_joint.legend_.set_title('Genetic Background') 
plt.suptitle('Joint distribution non-targeting and control Arid1a fl/fl organoids', fontsize=14)
# Ensure the folder exists
os.makedirs(folder_path, exist_ok=True)
# Save the figure to the specified path
#g.fig.savefig(folder_path +  "/joint_distribution_nontargeting_and_control_Arid1aflfl_organoids.pdf", dpi=300)
plt.show()


## 2. parental Arid1a -/- vs. transduced Arid1a -/- organoids with non-targeting sgRNAs
#Show the joint distribution using kernel density estimation
thickness_screen_checks_cre['GEMorganoid'] = pd.Categorical(thickness_screen_checks_cre['GEMorganoid'], categories=order, ordered=True)
g = sns.jointplot(
    data= thickness_screen_checks_cre,
    x="mean", y="std", hue="GEMorganoid",
    palette='Set2',  
)
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
g.ax_joint.legend_.set_title('Genetic Background') 
plt.suptitle('Joint distribution non-targeting and control Arid1a -/- organoids ', fontsize=14)
plt.tight_layout()
#g.fig.savefig(folder_path +  "/joint_distribution_nontargeting_and_control_Arid1aCRE_organoids.pdf", dpi=300)
plt.show()


thickness_screen_checks2 = thickness_measurements[thickness_measurements.date_sgRNA.str.endswith('controls') + thickness_measurements.GEMorganoid.str.startswith('NT_sgRNA')]
thickness_screen_checks2
thickness_screen_checks_NT = thickness_measurements[thickness_measurements.GEMorganoid.str.startswith('NT_sgRNA')]
display(thickness_screen_checks_NT)

In [ ]:
print (ks_2samp(
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'WT']['mean'], 
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'WT']['mean']
))

In [ ]:
print (ks_2samp(
    thickness_measurements_ctrls[thickness_measurements_ctrls.mmo1_line_x == 'WT']['std'], 
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'WT']['std']
))

Next we repeat the analysis for our true controls, the Arid1a fl/fl or Arid1a-/- organoid lines transduced with non-targeting sgRNAS. We check whether the distributions of 1. the mean ridge thickness, and 2. the standard deviation of the mean ridge thickness are different, respectively.

In [ ]:
#Layout of this plot
sns.set_style('ticks')

#Show the joint distribution using kernel density estimation
g = sns.jointplot(
    data= thickness_screen_checks_NT,
    x="mean", y="std", hue="mmo1_line",
    #kind="kde",
    palette='Set2',  
)
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
g.ax_joint.legend_.set_title('Genetic Background')  # Set the legend title to 'Genetic Background'
plt.suptitle('Joint distribution non-targeting Arid1a fl/fl vs. Arid1a -/- organoids', fontsize=14)

# Update legend labels
for t, l in zip(g.ax_joint.legend_.texts, legend_labels.values()):
    t.set_text(l)
    
plt.tight_layout()
#g.fig.savefig(folder_path +  "/joint_distribution_Arid1aflfl_and_Arid1aCRE_organoids.pdf", dpi=300)
plt.show

In [ ]:
print (ks_2samp(
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'WT']['mean'],
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'CRE']['mean']
))

In [ ]:
print (ks_2samp(
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'WT']['std'],
    thickness_screen_checks_NT[thickness_screen_checks_NT.mmo1_line == 'CRE']['std']
))

Conclusion: For both tests performed, the p-values are less than the significance level. 
We thereby reject the null hypotheses, concluding that there is a significant differences between the two distributions. 

### 4.B. Define Single-layered, cystic organoids 

Based on the above-performed analyses and based on the visual inspection of the pictures taken, we decided to set a ridge threshold to classify single-layered organoids in the dataset.
Therefore single-layered, organoids are organoids where the thickness average <14 and the SD of the thickness average < 6.

In [ ]:
THRESHOLD_THICKNESS = 14
THRESHOLD_STD = 6

#Layout of this plot
sns.set_style('ticks')

#Show the joint distribution using kernel density estimation
g = sns.jointplot(
    data= thickness_screen_checks_NT,
    x="mean", y="std", hue="mmo1_line",
    palette='Set2',  
)
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
g.ax_joint.legend_.set_title('Genetic Background') 
plt.suptitle('Joint distribution non-targeting Arid1a fl/fl vs. Arid1a -/- organoids', fontsize=14)
plt.axhline(y=THRESHOLD_STD, color='r', linestyle=':')
plt.axvline(x=THRESHOLD_THICKNESS, color='r', linestyle=':')

# Update legend labels
for t, l in zip(g.ax_joint.legend_.texts, legend_labels.values()):
    t.set_text(l)
    
#plt.tight_layout()
#g.fig.savefig(folder_path +  "/joint_distribution_NT_Arid1aflfl_and_Arid1aCRE_thresholds_organoids.pdf", dpi=300)
plt.show

Sanity check: perform the analysis on organoids that we classify as single-layered, cystic to see if we still see a significant differences between the two groups.

In [ ]:
single_layer_organoids = new_df[
    (new_df['mean'] < THRESHOLD_THICKNESS) & (new_df['std'] < THRESHOLD_STD)
]
display(single_layer_organoids)

# Count the number of entries with the same values for 'date_sgRNA', and 'mmo1_line'
count_single_layer_organoids = single_layer_organoids.groupby(['GEMorganoid', 'mmo1_line_x']).size().reset_index(name='count')
display(count_single_layer_organoids)
thickness_single_layer_organoids_ctrls = single_layer_organoids[single_layer_organoids.date_sgRNA.str.endswith('controls')]
sns.set_style('ticks')

# Show the joint distribution using kernel density estimation
g = sns.jointplot(
    data=thickness_single_layer_organoids_ctrls,
    x="mean", y="std", hue="mmo1_line_x",
    palette='Set2',  
)
g.set_axis_labels("Thickness of the Ridge (mean per organoid)", "Standard Deviation of the Ridge Thickness", fontsize=12)
g.ax_joint.legend_.set_title('Genetic Background')  
plt.suptitle('Joint distribution of single-layered, cystic organoids', fontsize=14)
# Update legend labels
for t, l in zip(g.ax_joint.legend_.texts, legend_labels.values()):
    t.set_text(l)
#plot data and save the figure    
plt.tight_layout()
#g.fig.savefig(folder_path +  "/joint_distribution_singlelayeredcystic_organoids.pdf", dpi=300)
plt.show()

In [ ]:
from scipy.stats import ks_2samp

print (ks_2samp(
    single_layer_organoids[single_layer_organoids.mmo1_line_x == 'WT']['mean'],
    single_layer_organoids[single_layer_organoids.mmo1_line_x == 'CRE']['mean']
))

In [ ]:
from scipy.stats import ks_2samp

print (ks_2samp(
    single_layer_organoids[single_layer_organoids.mmo1_line_x == 'WT']['std'],
    single_layer_organoids[single_layer_organoids.mmo1_line_x == 'CRE']['std']
))

### 4.C. Classify single-layered, organoids in the exisiting dataframe 

In the Qupath sofware, manual categorization was restricted to defining organoids as cystic, if these had one constant ridge. This part of the script aims to fine-tune the classification, and define single-layered cystic organoids based using thresholds previously identified and also in concordance with visual inspection of the dataset.

In [ ]:
# Define the condition
condition = (new_df['mean'] < THRESHOLD_THICKNESS) & (new_df['std'] < THRESHOLD_STD) & (new_df['Classification'] == 'cystic')
# Update 'Class' column where the condition is met
new_df.loc[condition, 'Classification'] = 'single-layered, cystic'
#display(new_df)

## 5. Phenotype distributions 

From the biological perspective it may not make sense to attribute a phenotype to organoids that are very small because they have not acquired defined phenotypes yet. We went into the images and looked at three different organoid size thresholds, in the upcoming script we filter out small organoids based on organoid area, to remove noise from the dataset. Three different organoid area sizes of 4500uM2, 10'000uM2 and 17'500uM2 were identified biologically.

In [ ]:
AREA_VALUES_TO_ITERATE_ON = [10000, 4500, 17500]

# Creating a loop to have all results be generated for each area threshold.
for i, area_threshold in enumerate(AREA_VALUES_TO_ITERATE_ON):
    print (f"Iteration number : {i}")
    print (f"Working with {area_threshold} area threshold ...") # allows to keep tract of which iteration the results have been generated from

    organoid_diameter = new_df[new_df['Area µm^2'] > area_threshold]
    print (f"Number of organoids passing the threshold : {organoid_diameter.shape[0]}")

    # Count the number of entries with the same values for 'date', 'organoid_id', and 'Classification'
    count_entries = organoid_diameter.groupby(['organoid_id', 'Classification', 'mmo1_line_x']).size().reset_index(name='count')
    count_entries = count_entries[count_entries['count'] > 0]
    display(count_entries)

    # Calculate the phenotype percentages
    phenotype_percentage  = 100 / organoid_diameter.groupby(['organoid_id', 'mmo1_line_x']).size() * organoid_diameter.groupby(['organoid_id','Classification', 'mmo1_line_x']).size()
    display(phenotype_percentage)

    # Merging counts and percentages.
    total_organoids = count_entries.set_index(['organoid_id', 'mmo1_line_x', 'Classification']).join(pd.DataFrame(phenotype_percentage).rename(columns={0: 'pheno_percent'}))
    total_organoids = total_organoids.reset_index()
    display(total_organoids)

    ## Create count plots
    df_wt = total_organoids[total_organoids['mmo1_line_x'] == 'WT']
    df_wt_copy = df_wt.copy()
  
    ##Create total count plots of organoid phenotypes
    stacked = df_wt[df_wt['organoid_id'] != 'controls']
    df_stacked = stacked[['organoid_id', 'Classification', 'count']].pivot_table(index='organoid_id', columns='Classification', values='count')
    display (df_stacked)


    #Plot total count WT organoids
    df_stacked.plot.bar( 
    stacked=True, 
    figsize=(20,10), 
    title= f'Total phenotype count Arid1a fl/fl organoids | threshold = {area_threshold}',
    ylabel ='Absolute organoid count',
    xlabel='', 
    color=['#4E29AC', '#AC2946', '#87AC29', '#29AC8F', '#FFC300'],
    )

    ## Absolute count for controls
    fig, ax = plt.subplots(1, 2, sharey=True)
    titles = ['WT', 'CRE']
    for i, temp_df in enumerate([df_wt]): #df_cre]):
        temp_stacked_df = temp_df[temp_df['organoid_id'] == 'controls']
        temp_stacked_df = temp_stacked_df[['organoid_id', 'Classification', 'count']].pivot_table(index='organoid_id', columns='Classification', values='count')
        temp_stacked_df.plot.bar(
            stacked=True, 
            title=f"{titles[i]}", 
            ax=ax[i], legend='Phenotypes',
            xlabel='',
        )
    ax[0].set_ylabel('Total phenotype count')
    legend = ax[i].legend(title='Phenotypes', loc='lower right')
    plt.tight_layout()
    sns.despine()
    #plt.savefig(folder_path + '/absolute_organoid_count.pdf', dpi=300)
    plt.show()

    ## Graph phenotype percentages 
    fig, ax = plt.subplots(2, 1, figsize=(20, 20))
    for i, cat in enumerate(['WT', 'CRE']):
        (
            total_organoids[
                total_organoids.mmo1_line_x == cat
            ][[
                'organoid_id', 
                'Classification', 
                'pheno_percent'
            ]]
            .pivot_table(
                index='organoid_id', 
                columns='Classification', 
                values='pheno_percent'
            )
            .fillna(0)
        ).plot.bar(
            stacked=True, 
            #cmap='Set3', # these are the princess palette colors
            color=['#4E29AC', '#AC2946', '#87AC29', '#29AC8F', '#FFC300'],
            ax=ax[i]
        )
        ax[i].set_title(cat)
        ax[i].set_xlabel('')  

        # Rename titles
        if cat == 'WT':
            ax[i].set_title('Arid1a fl/fl', fontsize=14)
        elif cat == 'CRE':
            ax[i].set_title('Arid1a -/-', fontsize=14)
        ax[i].set_ylabel('Phenotype distribution (%)')
        ax[i].legend(loc=(1.05, 0.5), title= f'Phenotypes | threshold = {area_threshold}')
    plt.tight_layout()
    sns.despine()
    #plt.savefig(folder_path + '/phenotype_percentages.pdf', dpi=300 )
    plt.show()
       
    total_organoids[(total_organoids.organoid_id == 'controls') & (total_organoids.mmo1_line_x == 'WT')]
    display(total_organoids)
    total_organoids_copy = total_organoids.copy()
    
    # Compute the Glandular to Cystic ratio 
    phenotypes_df_pivot = df_wt.pivot_table(
        index='organoid_id',
        columns='Classification',
        values='count',
    ).fillna(1)

    phenotypes_df_pivot['G_to_C']= phenotypes_df_pivot['glandular']/ (phenotypes_df_pivot['cystic'])
    phenotypes_df_pivot['G_to_IC']= phenotypes_df_pivot['glandular']/ (phenotypes_df_pivot['cystic'] + phenotypes_df_pivot['intermediate'])
    phenotypes_df_pivot['G_to_SLC']= phenotypes_df_pivot['glandular']/ (phenotypes_df_pivot['single-layered, cystic'])
    phenotypes_df_pivot['C_to_SLC']= (phenotypes_df_pivot['single-layered, cystic'])/phenotypes_df_pivot['cystic']
    phenotypes_df_pivot['SLC_to_G.C']= (phenotypes_df_pivot['single-layered, cystic'])/(phenotypes_df_pivot['cystic'] + phenotypes_df_pivot['glandular'])
    phenotypes_df_pivot['SLC_to_all']= (phenotypes_df_pivot['single-layered, cystic'])/(phenotypes_df_pivot['cystic'] + phenotypes_df_pivot['glandular'] + phenotypes_df_pivot['intermediate'])
    display(phenotypes_df_pivot)

    # Extracting organoid_id and G_to_SLC values
    organoid_id_wt = phenotypes_df_pivot.index
    C_to_slc_values_wt = phenotypes_df_pivot['C_to_SLC']
    C_to_slc_values_wt2 = phenotypes_df_pivot['SLC_to_G.C']
    C_to_slc_values_wt3 = phenotypes_df_pivot['SLC_to_all']
    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    plt.bar(organoid_id_wt, C_to_slc_values_wt, color='blue')
    plt.xlabel('Organoid ID')
    plt.ylabel('SLC_to_C Value')
    plt.title('SLC_to_C Values for Arid1a fl/fl organoids')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    #plt.savefig(folder_path + '/SLCtoC_values_Arid1aflfl_organoids.pdf', dpi=300)
    plt.show()

    # Create bar plot
    plt.figure(figsize=(14, 6))
    plt.bar(organoid_id_wt, C_to_slc_values_wt2, color='green')
    plt.xlabel('Organoid ID')
    plt.ylabel('SLC to G+C Value')
    plt.title('SLC to G+C Values for Arid1a fl/fl organoids')
    plt.xticks(rotation=90, ha='right')
    plt.tight_layout()
    #plt.savefig(folder_path + '/SLCtoGC_values_Arid1aflfl_organoids.pdf', dpi=300)
    plt.show()

    
    # Create bar plot
    plt.figure(figsize=(10, 6))
    plt.bar(organoid_id_wt, C_to_slc_values_wt3, color='red')
    plt.xlabel('Organoid ID')
    plt.ylabel('SLC to all Value')
    plt.title('SLC to all Values for Arid1a fl/fl organoids')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    #plt.savefig(folder_path + '/SLCtoall_values_Arid1aflfl_organoids.pdf', dpi=300)
    plt.show()

    df_cre = total_organoids[total_organoids['mmo1_line_x'] == 'CRE']
    phenotypes_cre_pivot = df_cre.pivot_table(
        index='organoid_id',
        columns='Classification',
        values='count',
    ).fillna(1)
    phenotypes_cre_pivot['G_to_C']= phenotypes_cre_pivot['glandular']/ (phenotypes_cre_pivot['cystic'])
    phenotypes_cre_pivot['G_to_IC']= phenotypes_cre_pivot['glandular']/ (phenotypes_cre_pivot['cystic'] + phenotypes_cre_pivot['intermediate'])
    phenotypes_cre_pivot['G_to_SLC']= phenotypes_cre_pivot['glandular']/ (phenotypes_cre_pivot['single-layered, cystic'])
    display(phenotypes_cre_pivot)

    # Extracting G_to_SLC values for each organoid ID
    organoid_ids_cre = phenotypes_cre_pivot.index
    g_to_slc_values_cre = [phenotypes_cre_pivot.loc[organoid_id, 'G_to_SLC'] for organoid_id in organoid_ids_cre]
    # Create bar plot
    plt.figure(figsize=(10, 6))
    plt.bar(organoid_ids_cre, g_to_slc_values_cre, color='blue')
    plt.xlabel('Organoid ID')
    plt.ylabel('G_to_SLC Value')
    plt.title('G_to_SLC Values for Arid1a -/- organoids')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    #plt.savefig(folder_path + '/GtoSLC_values_Arid1aCREorganoids.pdf', dpi=300)
    plt.show()
   
    # Define the different sets of classes
    class_sets = [
        ['cystic', 'single-layered, cystic', 'glandular', 'intermediate', 'undefined', 'undefinable'],
        ['cystic', 'single-layered, cystic'],
        ['cystic', 'single-layered, cystic', 'glandular']
    ]

    for i, classes in enumerate(class_sets, start=1):  
        # Define the different sets of classes, oop over the class sets and filter the DataFrame based on classes
        class_frequency = organoid_diameter[organoid_diameter['mmo1_line_x'] == 'WT'].loc[organoid_diameter['Classification'].isin(classes)].groupby(['organoid_id', 'mmo1_line_x'])['Classification'].value_counts(normalize=True) * 100
        class_frequency = class_frequency.rename('frequency_fraction').reset_index()

        # Calculate the total count for each combination of 'organoid_id', 'mmo1_line_x', and 'image_frame_x'
        total_counts = class_frequency.groupby(['organoid_id', 'mmo1_line_x'])['frequency_fraction'].sum()

        # Divide the frequency of each 'class' category by the total count
        class_frequency['frequency_fraction'] = class_frequency.apply(lambda row: row['frequency_fraction'] / total_counts[(row['organoid_id'], row['mmo1_line_x'])], axis=1)
        display(class_frequency)
        ordered_organoid_id = ['controls', 'NT_sgRNA1', 'NT_sgRNA2'] + sorted(
            value for value in class_frequency['organoid_id'].unique() if value not in ['controls', 'NT_sgRNA1', 'NT_sgRNA2']
        )
        plt.figure(figsize=(20,10))
        sns.barplot(
            data=class_frequency, 
            x='organoid_id',
            y='frequency_fraction', 
            hue='Classification', 
            order=ordered_organoid_id,
            palette=['r', 'b', 'g'],
            errorbar="sd",   
        )
        plt.xticks(rotation=90)
        # Set labels and title
        g.set_axis_labels("Organoid ID", "Mean Frequency Fraction")
        plt.title(f"Mean Frequency Fraction of Organoid Classes ({', '.join(classes)})")
        # Rotate x-axis labels
        plt.xticks(rotation=90)
        # Show plot
        #plt.savefig(folder_path +  "/Mean_frequencyfraction_of_organoid_classes.pdf", dpi=300)
        plt.show()
        
    break
   

Retrospectively: the 10'000 Size threshold was chosen as the most reliable one. A break was therefore added at the end of the code, to run it specifically on this size threshold.

## 6. Median Ridge Thickness of the GEM organoid line(s) - Statistics

We calculate the median thickness of the ridge for each of the genetically modified organoid lines, and assess whether the gene perturbations resulted in shifts in median ridge thickness of the population.

In [ ]:
Wgraph = organoid_diameter[(organoid_diameter['Classification'] == 'cystic') | (organoid_diameter['Classification'] == 'single-layered, cystic')]
Wgraph1 = Wgraph[Wgraph['mmo1_line_x'] == 'WT']
#display(Wgraph1)
# Define the order of 'organoid_id' column
custom_order = ['controls', 'NT_sgRNA1', 'NT_sgRNA2'] + sorted(
value for value in Wgraph1['organoid_id'].unique() if value not in ['controls', 'NT_sgRNA1', 'NT_sgRNA2']
)

# Initialize the figure with a logarithmic x axis
f, ax = plt.subplots(figsize=(10, 16))
PROPS = {
'boxprops':{'facecolor':'none', 'edgecolor':'black'},
'medianprops':{'color':'black'},
'whiskerprops':{'color':'black'},
'capprops':{'color':'black'},   
}

# Plot the orbital period with horizontal boxes
sns.boxplot(
    data=Wgraph1, 
    x="mean", 
    y="organoid_id", 
    order=custom_order,            
    showfliers=False,
    linewidth=1,
    ax=ax,
    **PROPS,
    whis=[0, 100], 
    width=.6,
)

# Add in points to show each observation
sns.stripplot( data=Wgraph1, x="mean", y="organoid_id", size=4, hue='Classification', order=custom_order) 
# Tweak the visual presentation
ax.xaxis.grid(True)
ax.set(ylabel="Conditions")
ax.set(xlabel="Organoid ridge thickness (um)")
sns.despine(trim=True, left=True)

# Save the plot to the specified path
plt.tight_layout()
#plt.savefig(folder_path + '/Median_organoid_ridge_thicknessses_allconditions.pdf', dpi=300)
# Show the plot
plt.show()

In the following code, we perform loop over the data and each time perform a Mann-Whitney U rank test on two independent samples. Meaning, we compare the non-targeting control(s) to a respective genetically modified organoid line, and assess whether the distributions differ based on the introduced gene knockout. This nonparametric test was chosen with respects to the previously performed Shapiro-Wilk test results.

In [ ]:
##Mann whitney U of one median versus the one of NT_sgRNA1 
from scipy.stats import mannwhitneyu

# Initialize a list to hold the results
results = []

# Filter the DataFrame to select rows with 'organoid_id' equal to 'NT_sgRNA1'
nt_sgRNA1_df = Wgraph1[Wgraph1['organoid_id'] == 'NT_sgRNA1']
# Calculate the median of 'mean' values for 'NT_sgRNA1'
nt_sgRNA1_median = nt_sgRNA1_df['mean'].median()

# Iterate through each unique 'organoid_id' value other than 'NT_sgRNA1'
for organoid_id in Wgraph1['organoid_id'].unique():
    if organoid_id == 'NT_sgRNA1':
        continue  # Skip 'NT_sgRNA1' as we have already calculated its median
    # Filter the DataFrame to select rows with the current 'organoid_id'
    current_organoid_df = Wgraph1[Wgraph1['organoid_id'] == organoid_id]
    # Calculate the median of 'mean' values for the current 'organoid_id'
    current_organoid_median = current_organoid_df['mean'].median()
    # Perform the Mann-Whitney U test
    statistic, p_value = mannwhitneyu(nt_sgRNA1_df['mean'], current_organoid_df['mean'])
    
    # Print the results
    print(f"Comparing {organoid_id} to NT_sgRNA1:")
    print(f"Median of NT_sgRNA1: {nt_sgRNA1_median}")
    print(f"Median of {organoid_id}: {current_organoid_median}")
    print(f"Mann-Whitney U statistic: {statistic}")
    print(f"P-value: {p_value}")
    if p_value < 0.05:
        print("There is a SIGNIFICANT difference between the medians of the two groups.")
    else:
        print("There is no significant difference between the medians of the two groups,")
    print()

    # Store the results
    result = {
        "Comparison": f"{organoid_id} vs NT_sgRNA1",
        "Median NT_sgRNA1": nt_sgRNA1_median,
        "Median other group": current_organoid_median,
        "Mann-Whitney U statistic": statistic,
        "P-value": p_value,
        "Significance": "SIGNIFICANT difference" if p_value < 0.05 else "No significant difference"
    }
    results.append(result)

# Convert the results list to a DataFrame
results_df = pd.DataFrame(results)

# Define the file path
file_path = '/Users/deby/Desktop/lentiscreen_data/4_Data/4.3_Results_statistics/mannwhitneyu_toNTsgRNA1.csv'

# Save the DataFrame to a CSV file
#results_df.to_csv(file_path, index=False)

file_path

In [ ]:
##Mann whitney U of one median versus the one of NT_sgRNA2 
from scipy.stats import mannwhitneyu

# Initialize a list to hold the results
results = []

# Filter the DataFrame to select rows with 'organoid_id' equal to 'NT_sgRNA2'
nt_sgRNA2_df = Wgraph1[Wgraph1['organoid_id'] == 'NT_sgRNA2']
# Calculate the median of 'mean' values for 'NT_sgRNA2'
nt_sgRNA2_median = nt_sgRNA2_df['mean'].median()

# Iterate through each unique 'organoid_id' value other than 'NT_sgRNA2'
for organoid_id in Wgraph1['organoid_id'].unique():
    if organoid_id == 'NT_sgRNA2':
        continue  # Skip 'NT_sgRNA2' as we have already calculated its median
    # Filter the DataFrame to select rows with the current 'organoid_id'
    current_organoid_df = Wgraph1[Wgraph1['organoid_id'] == organoid_id]
    # Calculate the median of 'mean' values for the current 'organoid_id'
    current_organoid_median = current_organoid_df['mean'].median()
    # Perform the Mann-Whitney U test
    statistic, p_value = mannwhitneyu(nt_sgRNA2_df['mean'], current_organoid_df['mean'])
    
    # Print the results
    print(f"Comparing {organoid_id} to NT_sgRNA2:")
    print(f"Median of NT_sgRNA2: {nt_sgRNA2_median}")
    print(f"Median of {organoid_id}: {current_organoid_median}")
    print(f"Mann-Whitney U statistic: {statistic}")
    print(f"P-value: {p_value}")
    if p_value < 0.05:
        print("There is a SIGNIFICANT difference between the medians of the two groups.")
    else:
        print("There is no significant difference between the medians of the two groups,")
    print()

    # Store the results
    result = {
        "Comparison": f"{organoid_id} vs NT_sgRNA2",
        "Median NT_sgRNA2": nt_sgRNA2_median,
        "Median other group": current_organoid_median,
        "Mann-Whitney U statistic": statistic,
        "P-value": p_value,
        "Significance": "SIGNIFICANT difference" if p_value < 0.05 else "No significant difference"
    }
    results.append(result)

# Convert the results list to a DataFrame
results_df = pd.DataFrame(results)

# Define the file path
file_path = '/Users/deby/Desktop/lentiscreen_data/4_Data/4.3_Results_statistics/mannwhitneyu_toNTsgRNA2.csv'

# Save the DataFrame to a CSV file
#results_df.to_csv(file_path, index=False)

file_path


In [ ]:
## Mann whitney U of one median versus the one of (NT_sgRNA1 + NT_sgRNA2)

from scipy.stats import mannwhitneyu

# Initialize a list to hold the results
results = []

# Filter the DataFrame to select rows with 'organoid_id' equal to 'NT_sgRNA1' and 'NT_sgRNA2'
nt_sgRNAs_df = Wgraph1[Wgraph1['organoid_id'].isin(['NT_sgRNA1', 'NT_sgRNA2'])]
# Calculate the median of 'mean' values for 'NT_sgRNAs'
nt_sgRNAs_median = nt_sgRNAs_df['mean'].median()

# Iterate through each unique 'organoid_id' value other than 'NT_sgRNAs'
for organoid_id in Wgraph1['organoid_id'].unique():
    if organoid_id in ['NT_sgRNA1', 'NT_sgRNA2']:
        continue  # Skip 'NT_sgRNA1' and 'NT_sgRNA2' as we have already calculated their median
    # Filter the DataFrame to select rows with the current 'organoid_id'
    current_organoid_df = Wgraph1[Wgraph1['organoid_id'] == organoid_id]
    # Calculate the median of 'mean' values for the current 'organoid_id'
    current_organoid_median = current_organoid_df['mean'].median()
    
    # Perform the Mann-Whitney U test
    statistic, p_value = mannwhitneyu(nt_sgRNAs_df['mean'], current_organoid_df['mean'])
    
    # Print the results
    #print(nt_sgRNAs_df['mean'].isnull().sum())
    #print(current_organoid_df['mean'].isnull().sum())

    print(f"Comparing {organoid_id} to NT_sgRNAs:")
    print(f"Median of NT_sgRNAs: {nt_sgRNAs_median}")
    print(f"Median of {organoid_id}: {current_organoid_median}")
    print(f"Mann-Whitney U statistic: {statistic}")
    print(f"P-value: {p_value}")
    if p_value < 0.05:
        print("There is a SIGNIFICANT difference between the medians of the two groups.")
    else:
        print("There is no significant difference between the medians of the two groups.")
    print()

     # Store the results
    result = {
        "Comparison": f"{organoid_id} vs NT_sgRNAs",
        "Median NT_sgRNAs": nt_sgRNA2_median,
        "Median other group": current_organoid_median,
        "Mann-Whitney U statistic": statistic,
        "P-value": p_value,
        "Significance": "SIGNIFICANT difference" if p_value < 0.05 else "No significant difference"
    }
    results.append(result)

# Convert the results list to a DataFrame
results_df = pd.DataFrame(results)

# Define the file path
file_path = '/Users/deby/Desktop/lentiscreen_data/4_Data/4.3_Results_statistics/mannwhitneyu_toNTsgRNAs.csv'

# Save the DataFrame to a CSV file
#results_df.to_csv(file_path, index=False)

file_path


## 7. Visualizing phenotype shifts

In the following section we visualize phenotype shifts by checking how a given genetic perturbation affects both the median thickness of the ridge of cystic organoids and the number of single-layered cystic organoid versus other phenotypes in the culture.

In [ ]:
xx =  Wgraph1.groupby(['organoid_id'])['mean'].median()
display(xx)
xx = xx.reset_index()

In [ ]:
# Merge the pivot table with the table of SLC_to_GC ratios
merged_df2 = pd.merge(xx, C_to_slc_values_wt2, on='organoid_id', how='left')
merged_df2 = merged_df2[merged_df2['organoid_id'] != 'controls']
# Display the merged DataFrame
display(merged_df2)
#Note: Here 'mean' refers to median thickness of the populations, with each point being the mean organoid thickness of an individual organoid

In [ ]:
#Create a bokeh plot to hover over the data, and have an interactive data inspection tool
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource

# Convert DataFrame to ColumnDataSource
source = ColumnDataSource(merged_df2)

# Create a Bokeh figure
p = figure(title='Scatter Plot', x_axis_label='Median thickness of the ridge', y_axis_label='Cystic_to_SLC organoid ratio')

# Add scatter points
p.scatter('mean', 'SLC_to_G.C', source=source, legend_field='organoid_id', fill_alpha=0.6, size=10, color='red')

# Add hover tool
hover = HoverTool(tooltips=[('Organoid ID', '@organoid_id')])
p.add_tools(hover)
# Show the plot
show(p)

In [ ]:

plt.figure(figsize=(12, 10))

# Create a scatter plot 
sns.scatterplot(data=merged_df2, x='mean', y='SLC_to_G.C', color='red')

#colouring was done based on inspection of the statistical results.
# Filter the data to get the points that meet set stat. conditions
filtered_data = merged_df2[(merged_df2['mean'] > 16.5) | (merged_df2['SLC_to_G.C'] < 0.30)]
# Add scatter points for data points that meet the conditions
sns.scatterplot(data=filtered_data, x='mean', y='SLC_to_G.C', color='grey')

# Highlight non-targeting controls in blue
highlighted_data = merged_df2[(merged_df2['organoid_id'] == 'NT_sgRNA1') | (merged_df2['organoid_id'] == 'NT_sgRNA2')]
sns.scatterplot(data=highlighted_data, x='mean', y='SLC_to_G.C', color='blue', s=40)

# Add text annotations for organoid IDs for points that don't meet the conditions
prev_x, prev_y = None, None
for organoid_id, mean, slc_to_gc in zip(merged_df2['organoid_id'], merged_df2['mean'], merged_df2['SLC_to_G.C']):
    if (mean <= 16.5) and (slc_to_gc >= 0.30):
        if prev_x is None or abs(prev_x - mean) > 0.2 or abs(prev_y - slc_to_gc) < 0.15:
            plt.text(x=mean + 0.1, y=slc_to_gc, s=str(organoid_id), fontsize=10)
            prev_x, prev_y = mean + 0.1, slc_to_gc
        else:
            plt.text(x=mean + 0.1, y=prev_y + 0.15, s=str(organoid_id), fontsize=10)
            prev_y += 0.15

# Set labels and title
plt.xlabel('Median thickness of the ridge')
plt.ylabel('SLC_to_G.C organoid ratio')
plt.title('Scatter Plot')
plt.tight_layout()

#plt.savefig(folder_path + '/SLC:GCratio_verus_medianthickness_bothNTsgRNAs.pdf', dpi=300)

The graph above is repeated but now pooling all non-targeting controls in one datapoint. For this some upstream analysis needs to be reperformed on the dataframes.

In [ ]:
organoid_diamteter_copy = organoid_diameter.copy() #copying of the dataframe to not overwrite or mess with exisiting results 

# Count the number of entries with the same values for 'date', 'organoid_id', and 'Classification'
count_entries2 = organoid_diamteter_copy.groupby(['organoid_id', 'Classification', 'mmo1_line_x']).size().reset_index(name='count')
count_entries2 = count_entries2[count_entries2['count'] > 0]
#display(count_entries2)

# Merging coutns and percentages.
total_organoids2 = count_entries2.set_index(['organoid_id', 'mmo1_line_x', 'Classification']).join(pd.DataFrame(phenotype_percentage).rename(columns={0: 'pheno_percent'}))
total_organoids2 = total_organoids2.reset_index()
#display(total_organoids2)

## Create count plots
df_wt_2 = total_organoids2[total_organoids2['mmo1_line_x'] == 'WT']
#display(df_wt_2)
df_wt_2.rename(columns={'organoid_id': 'GEMorganoid'}, inplace=True)
df_wt_2['GEMorganoid'] =  df_wt_2['GEMorganoid'].replace({'NT_sgRNA2': 'NT_sgRNA', 'NT_sgRNA1': 'NT_sgRNA'})
#display(df_wt_2)
# Compute the Glandular to Cystic ratio 
phenotypes_df_pivot2 = df_wt_2.pivot_table(
    index='GEMorganoid',
    columns='Classification',
    values='count',
).fillna(1)
#display(phenotypes_df_pivot2)
phenotypes_df_pivot2['SLC_to_G.C']= (phenotypes_df_pivot2['single-layered, cystic'])/(phenotypes_df_pivot2['cystic'] + phenotypes_df_pivot2['glandular'])
display(phenotypes_df_pivot2)
# Extracting organoid_id and G_to_SLC values
C_to_slc_values_wt2 = phenotypes_df_pivot2['SLC_to_G.C']
print("Resulting C_to_slc_values_wt2 dataframe:")
display(C_to_slc_values_wt2)

In [ ]:
# Replace 'NT_sgRNA2' and 'NT_sgRNA1' with 'NT_sgRNA' and save as a new DataFrame
Wgraph1_modified = Wgraph1.copy() #copying of the dataframe to not overwrite or mess with exisiting results 
Wgraph1_modified['GEMorganoid'] = Wgraph1_modified['GEMorganoid'].replace({'NT_sgRNA2': 'NT_sgRNA', 'NT_sgRNA1': 'NT_sgRNA'})
#display(Wgraph1_modified)
Wgraph_modified2 = Wgraph1_modified[Wgraph1_modified['organoid_id'] != 'controls'] #filter out the parental cell lines

# Check for NT_sgRNA in the modified DataFrame
nt_sgrna_rows = Wgraph_modified2[Wgraph_modified2['GEMorganoid'] == 'NT_sgRNA']
# Display the rows containing NT_sgRNA
#display(nt_sgrna_rows)

yy =  Wgraph_modified2.groupby(['GEMorganoid'])['mean'].median()
#display(yy)

# Use reset_index() to convert the result of groupby to a DataFrame
yy = yy.reset_index()

# Merge the pivot table with the table of SLC_to_G.C ratios
merged_df3 = pd.merge(yy, C_to_slc_values_wt2, on='GEMorganoid', how='left')
print("the 'mean' title of the column represents the median ridge thickness of the organoid population")
display(merged_df3)

In [ ]:
# Graph version 2
sns.set_theme('talk')
sns.set_style('ticks')
plt.figure(figsize=(10,8))
sns.scatterplot(data=merged_df3, x='mean', y='SLC_to_G.C', color='red', s=140)

# Filter the data to get the points that the statistical conditions.
filtered_data2 = merged_df3[(merged_df3['mean'] > 16.5) | (merged_df3['SLC_to_G.C'] < 0.30)]
# Add scatter points for data points that meet the conditions
sns.scatterplot(data=filtered_data2, x='mean', y='SLC_to_G.C', color='grey', s=140)

# Highlight the non-targeting control in blue
highlighted_data = merged_df3[merged_df3['GEMorganoid'] == 'NT_sgRNA']
sns.scatterplot(data=highlighted_data, x='mean', y='SLC_to_G.C', color='blue', s=140)

# Add text annotations for organoid IDs for points that don't meet the conditions
prev_x, prev_y = None, None
for GEMorganoid, mean, slc_to_gc in zip(merged_df3['GEMorganoid'], merged_df3['mean'], merged_df3['SLC_to_G.C']):
    if (mean <= 16.5) and (slc_to_gc >= 0.30):
        if prev_x is None or abs(prev_x - mean) > 0.2 or abs(prev_y - slc_to_gc) < 0.15:
            plt.text(x=mean + 0.1, y=slc_to_gc, s=str(GEMorganoid), fontsize=14)
            prev_x, prev_y = mean + 0.1, slc_to_gc
        else:
            plt.text(x=mean + 0.1, y=prev_y + 0.15, s=str(GEMorganoid), fontsize=14)
            prev_y += 0.15

# Set labels and title
plt.xlabel('Median thickness of the ridge', fontsize=16)
plt.ylabel('SLC_to_G.C organoid ratio', fontsize=16)
plt.title('Scatter Plot')
plt.tight_layout()

#plt.savefig(folder_path + '/SLC:GCratio_verus_medianthickness_pooledNTsgRNAs.pdf', dpi=300)

## 8. Fisher Exact Statistics 

In [ ]:
df_stacked_copy = df_wt_copy[['organoid_id', 'Classification', 'count']].pivot_table(index='organoid_id', columns='Classification', values='count')
display (df_stacked_copy)

In [ ]:
phenotypes_cre_pivot

In [ ]:
## FISHER EXACT, for 10'000 comparing parental organoid lines
# importing packages 
import scipy.stats as stats 

# creating data 
WT_SLC = df_stacked_copy.loc['controls', 'single-layered, cystic']
WT_GC = df_stacked_copy.loc['controls', 'glandular' ] + df_stacked_copy.loc['controls', 'cystic']
CRE_SLC = phenotypes_cre_pivot.loc['controls', 'single-layered, cystic']
CRE_GC = phenotypes_cre_pivot.loc['controls', 'glandular' ] + phenotypes_cre_pivot.loc['controls', 'cystic']

data = [[WT_SLC, WT_GC], [CRE_SLC, CRE_GC]]
odds_ratio, p_value = stats.fisher_exact(data)

# performing fishers exact test on the data 
odd_ratio, p_value = stats.fisher_exact(data) 
print('odd ratio for parental Arid1a fl/fl vs Arid1a -/- is : ' + str(odd_ratio)) 
print('p_value for parental Arid1a fl/fl vs Arid1a -/- is : ' + str(p_value)) 

## FISHER EXACT, for NT_sgRNAs Arid1a fl/fl vs NT_sgRNAs Arid1a -/-
# creating data 
NT_WT_SLC = df_stacked_copy.loc['NT_sgRNA1', 'single-layered, cystic'] + df_stacked_copy.loc['NT_sgRNA2', 'single-layered, cystic']
NT_WT_GC = df_stacked_copy.loc['NT_sgRNA1', 'glandular' ] + df_stacked_copy.loc['NT_sgRNA1', 'cystic'] + df_stacked_copy.loc['NT_sgRNA2', 'glandular' ] + df_stacked_copy.loc['NT_sgRNA2', 'cystic']
NT_CRE_SLC = phenotypes_cre_pivot.loc['NT_sgRNA1', 'single-layered, cystic'] + phenotypes_cre_pivot.loc['NT_sgRNA2', 'single-layered, cystic']
NT_CRE_GC = phenotypes_cre_pivot.loc['NT_sgRNA1', 'glandular' ] + phenotypes_cre_pivot.loc['NT_sgRNA1', 'cystic'] + phenotypes_cre_pivot.loc['NT_sgRNA2', 'glandular'] + phenotypes_cre_pivot.loc['NT_sgRNA2', 'cystic']

data2 = [[NT_WT_SLC, NT_WT_GC], [NT_CRE_SLC, NT_CRE_GC]]
odds_ratio2, p_value2 = stats.fisher_exact(data2)

# performing fishers exact test on the data 
odd_ratio2, p_value2 = stats.fisher_exact(data2) 
print('odd ratio for Arid1a fl/fl NT vs Arid1a -/- NT is : ' + str(odd_ratio2)) 
print('p_value for Arid1a fl/fl NT vs Arid1a -/- NT is : ' + str(p_value2)) 

# Create a DataFrame to store the results
results = pd.DataFrame({
    'Fishers exact test': ['parental organoid lines', 'parental organoid lines transduced with non-targeting sgRNAs (NT_sgRNAs)'],
    'Odds Ratio': [odds_ratio, odds_ratio2],
    'P-Value': [p_value, p_value2]
})

# Save the results to a CSV file
#results.to_csv(folder_path_stats + '/fisher_exact_results.csv', index=False)

print("Results saved to 'fisher_exact_results.csv'")

### 8.A. Fisher extact statistics for SLC vs G+C counts

To test for the association between genotype and phenotype distributions we performed a Fisher's exact test. This non-parametric test was chose based on the Shapiro test result performed earlier. The null hypothesis of no association between the two categorical variables is tested against the alternate hypothesis that there is an association between the two variables. We test this hypothesis for every gene knockout performed, and output the statistical results into csv files for inspection.

In [ ]:
"""# The final statistics are performed based on an organoid size threshold of 10000uM2
area_values_to_iterate_on = [10000]

# Creating a loop to have all results be generated for each area threshold.
for i, area_threshold in enumerate(area_values_to_iterate_on):
    print (f"Iteration number : {i}")
    print (f"Working with {area_threshold} area threshold ...") # allows me to keep tract of which iteration my results have been generated from

    organoid_diameter = new_df[new_df['Area µm^2'] > area_threshold]
    print (f"Number of organoids passing the threshold : {organoid_diameter.shape[0]}")

    # Count the number of entries with the same values for 'date', 'organoid_id', and 'Classification'
    count_entries2 = organoid_diameter.groupby(['organoid_id', 'Classification', 'mmo1_line_x']).size().reset_index(name='count')
    count_entries2 = count_entries[count_entries['count'] > 0]
    #display(count_entries)
    
    ## Create count plots
    df_wt =  count_entries2[count_entries2['mmo1_line_x'] == 'WT']
  
    ##Create total count plots of organoid phenotypes
    stacked = df_wt[df_wt['organoid_id'] != 'controls']
    df_stacked = stacked[['organoid_id', 'Classification', 'count']].pivot_table(index='organoid_id', columns='Classification', values='count').fillna(0)

    # Selecting desired columns and renaming them
    stats_wt = df_stacked[['cystic', 'single-layered, cystic', 'glandular']].reset_index()

    stats_wt.columns = ['organoid_id', 'cystic', 'single-layered, cystic', 'glandular']
    #Displaying the new dataframe
    display(stats_wt)
 

    # Assuming the DataFrame is named df
    organoid_ids_pp = stats_wt['organoid_id'].tolist()

    # Print the list of organoid_id values
    print(organoid_ids_pp)
    break
"""

In [ ]:
AREA_THRESHOLD = 10000
organoid_diameter = new_df[new_df['Area µm^2'] > AREA_THRESHOLD]

count_wt = (
    organoid_diameter
    .groupby(['organoid_id', 'Classification', 'mmo1_line_x'])
    .size()
    .reset_index(name='count')
)

df_wt = count_wt[count_wt['mmo1_line_x'] == 'WT']
stacked = df_wt[df_wt['organoid_id'] != 'controls']

stats_wt = (
    stacked
    .pivot_table(index='organoid_id', columns='Classification', values='count')
    .fillna(0)[
        ['cystic', 'single-layered, cystic', 'glandular']
    ]
    .reset_index()
)
stats_wt['g+c'] = stats_wt['cystic'] + stats_wt['glandular']

In [ ]:
#stats_wt['g+c'] = stats_wt['cystic'] + stats_wt['glandular']
display(stats_wt)
# Set 'organoid_id' as the index
stats_wt_copy = stats_wt.copy()
stats_wt_copy.set_index('organoid_id', inplace=True)
display(stats_wt_copy)

In [ ]:
NT_SLC = stats_wt_copy.loc['NT_sgRNA1', 'single-layered, cystic']  + stats_wt_copy.loc['NT_sgRNA2', 'single-layered, cystic']

In [ ]:
from scipy.stats import fisher_exact
## fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA1nd2_results
print('fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA1nd2_results')

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT_GC, NT_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['g+c']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT_GC], [SLC_value, NT_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

#Create the data
NT_SLC = stats_wt_copy.loc['NT_sgRNA1', 'single-layered, cystic']  + stats_wt_copy.loc['NT_sgRNA2', 'single-layered, cystic']
NT_GC = stats_wt_copy.loc['NT_sgRNA1', 'g+c' ] + stats_wt_copy.loc['NT_sgRNA2', 'g+c']
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt, NT_GC, NT_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoGCratio__ARID1Aflflorganoids_to_NTsgRNA1nd2_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")


In [ ]:
## fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA1_results
print('fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA1_results')

from scipy.stats import fisher_exact

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT1_GC , NT1_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['g+c']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT1_GC ], [SLC_value, NT1_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

#Create the data
NT1_SLC = stats_wt_copy.loc['NT_sgRNA1', 'single-layered, cystic'] 
NT1_GC = stats_wt_copy.loc['NT_sgRNA1', 'g+c' ] 
print(NT1_SLC)
print(NT1_GC)
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt, NT1_GC , NT1_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoGCratio_ARID1Aflflorganoids_to_NTsgRNA1_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")



In [ ]:
print('fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA2_results')
## fisher_exact_test_ARID1Aflflorganoids_to_NTsgRNA2_results
from scipy.stats import fisher_exact

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT2_GC, NT2_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['g+c']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT2_GC], [SLC_value, NT2_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

#Create the data
NT2_SLC = stats_wt_copy.loc['NT_sgRNA2', 'single-layered, cystic'] 
NT2_GC = stats_wt_copy.loc['NT_sgRNA2', 'g+c' ] 
print(NT2_SLC)
print(NT2_GC)
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt, NT2_GC, NT2_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoGCratio_ARID1Aflflorganoids_to_NTsgRNA2_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")


### 8.B. Fisher extact statistics for SLC vs all (other organoid types) counts 

In [ ]:
"""
area_values_to_iterate_on = [10000]

# results be generated for the desired threshold.
for i, area_threshold in enumerate(area_values_to_iterate_on):
    print (f"Iteration number : {i}")
    print (f"Working with {area_threshold} area threshold ...") 

    organoid_diameter = new_df[new_df['Area µm^2'] > area_threshold]
    print (f"Number of organoids passing the threshold : {organoid_diameter.shape[0]}")

    # Count the number of entries with the same values for 'date', 'organoid_id', and 'Classification'
    count_entries2_all = organoid_diameter.groupby(['organoid_id', 'Classification', 'mmo1_line_x']).size().reset_index(name='count')
    count_entries2_all = count_entries[count_entries['count'] > 0]
  
    ## Create count plots
    df_wt_all =  count_entries2_all[count_entries2_all['mmo1_line_x'] == 'WT']
    
    ##Create total count plots of organoid phenotypes
    stacked_all = df_wt_all[df_wt_all['organoid_id'] != 'controls']
    df_stacked_all = stacked[['organoid_id', 'Classification', 'count']].pivot_table(index='organoid_id', columns='Classification', values='count').fillna(0)
    #display (df_stacked)

    # Selecting desired columns and renaming them
    stats_wt_all = df_stacked_all[['cystic', 'single-layered, cystic', 'intermediate', 'glandular']].reset_index()

    stats_wt.columns = ['organoid_id', 'cystic', 'single-layered, cystic', 'glandular', 'intermediate']
    #Displaying the new dataframe
    display(stats_wt_all)
 
    break

# Assuming your DataFrame is named df
stats_wt_all['all'] = stats_wt_all['cystic'] + stats_wt_all['glandular'] + stats_wt_all['intermediate']

# Display the updated DataFrame
display(stats_wt_all)
"""

In [ ]:
count_all = (
    organoid_diameter
    .groupby(['organoid_id', 'Classification', 'mmo1_line_x'])
    .size()
    .reset_index(name='count')
)
df_wt_all = count_all[count_all['mmo1_line_x'] == 'WT']
stacked_all = df_wt_all[df_wt_all['organoid_id'] != 'controls']

stats_wt_all = (
    stacked_all
    .pivot_table(index='organoid_id', columns='Classification', values='count')
    .fillna(0)[
        ['cystic', 'single-layered, cystic', 'intermediate', 'glandular']
    ]
    .reset_index()
)
stats_wt_all['all'] = (
    stats_wt_all['cystic']
    + stats_wt_all['glandular']
    + stats_wt_all['intermediate']
)

display(stats_wt_all)

In [ ]:
stats_wt_all_copy= stats_wt_all.copy()

stats_wt_all_copy.index = pd.Series(stats_wt_all.organoid_id)
stats_wt_all_copy

In [ ]:
print("comparing ARID1A FL/FL organoids to NT_sgRNA1+2")
## comparing ARID1A FL/FL organoids to ARID1A FL/FL CONTROL
from scipy.stats import fisher_exact

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT_all , NT_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['all']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT_all], [SLC_value, NT_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

# Fixed variables for the test
NT_all = stats_wt_all_copy.loc['NT_sgRNA1', 'all'] + stats_wt_all_copy.loc['NT_sgRNA2', 'all']
print(NT_all)
NT_SLC 
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt_all, NT_all , NT_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoALLratio_ARID1Aflflorganoids_to_NTsgRNAs_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt_all.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")

In [ ]:
print("comparing ARID1A FL/FL organoids to NT_sgRNA1")

## comparing ARID1A FL/FL organoids to ARID1A FL/FL CONTROL
from scipy.stats import fisher_exact

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT1_all, NT1_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['all']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT1_all], [SLC_value, NT1_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

# Fixed variables for the test
NT1_all = stats_wt_all_copy.loc['NT_sgRNA1', 'all'] 
print(NT1_all)
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt_all, NT1_all, NT1_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoALLratio_ARID1Aflflorganoids_to_NTsgRNA1_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt_all.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")

In [ ]:
print("comparing ARID1A FL/FL organoids to NT_sgRNA2")
## comparing ARID1A FL/FL organoids to ARID1A FL/FL CONTROL
from scipy.stats import fisher_exact

# Define the function to perform Fisher's exact test
def fisher_exact_test_loop(df, NT2_all, NT2_SLC):
    results = []
    for index, row in df.iterrows():
        glandular_value = row['all']
        SLC_value = row['single-layered, cystic']
        contingency_table = [[glandular_value, NT2_all], [SLC_value, NT2_SLC]]
        _, p_value = fisher_exact(contingency_table)
        results.append((index, row['organoid_id'], p_value))
    return results

# Data
NT2_all = stats_wt_all_copy.loc['NT_sgRNA2', 'all']
NT2_SLC 
alpha = 0.05

# Run Fisher's exact test
results = fisher_exact_test_loop(stats_wt_all, NT2_all, NT2_SLC)

# Create a DataFrame from results
results_df = pd.DataFrame(results, columns=['Index', 'Organoid ID', 'P-Value'])

# Add significance column
results_df['Significant'] = results_df['P-Value'] < alpha

# Save results to CSV
#results_df.to_csv(folder_path_stats + '/fisher_exact_test_SLCtoALLratio_ARID1Aflflorganoids_to_NTsgRNA2_results.csv', index=False)

# Print results to console
for index, organoid_id, p_value in results:
    organoid_id = stats_wt_all.loc[index, 'organoid_id']
    print(f"For row {organoid_id} | {index}: p-value = {p_value} | ", end="")
    if p_value < alpha:
        print("significant.")
    else:
        print("no association.")

## 9.Appendix: Sanity check - for outlier images in the screen positive controls

In [ ]:
import pandas as pd
from scipy.stats import f_oneway
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Given that small organoids have not acquired defined phenotypes yet, filter out on organoid area.
area_values_to_iterate_on = [10000]

# Loop to generate results for each area threshold.
for i, area_threshold in enumerate(area_values_to_iterate_on):
    print(f"Iteration number: {i}")
    print(f"Working with {area_threshold} area threshold...")

    organoid_diameter = new_df[new_df['Area µm^2'] > area_threshold]
    print(f"Number of organoids passing the threshold: {organoid_diameter.shape[0]}")

    # Filter for specific GEMorganoids
    Arid1a_entries = organoid_diameter[organoid_diameter['GEMorganoid'].isin(['Arid1a_sgRNA1', 'Arid1a_sgRNA2', 'Arid1a_sgRNA3'])]

    # Count the number of entries with the same values for selected columns
    count_entries_A = Arid1a_entries.groupby(['GEMorganoid', 'image_frame_y', 'Classification', 'mean']).size().reset_index(name='count')

    # Pivot to get counts for 'single-layered, cystic' and 'cystic'
    pivot_counts = count_entries_A.pivot_table(index=['GEMorganoid', 'image_frame_y'], columns='Classification', values='count', fill_value=0)
    
    # Calculate the 'C_to_SLC' ratio
    pivot_counts['C_to_SLC'] = pivot_counts['cystic'] / pivot_counts['single-layered, cystic']
    
    # Handle cases where 'single-layered, cystic' count is 0
    pivot_counts['C_to_SLC'].replace([float('inf'), -float('inf')], 0, inplace=True)
    pivot_counts['C_to_SLC'].fillna(0, inplace=True)
    
    # Reset index to merge back to the original DataFrame
    pivot_counts.reset_index(inplace=True)

    # Merge 'C_to_SLC' ratio back to the original DataFrame
    df_with_counts = pd.merge(count_entries_A, pivot_counts[['GEMorganoid', 'image_frame_y', 'C_to_SLC']], on=['GEMorganoid', 'image_frame_y'], how='left')

    # Bin the C_to_SLC values
    bins = np.arange(0, 2.25, 0.25)
    df_with_counts['C_to_SLC_binned'] = pd.cut(df_with_counts['C_to_SLC'], bins=bins, include_lowest=True)

    # Debugging: Print the DataFrame to check columns
    print("DataFrame before plotting:")
    print(df_with_counts)

    # Plotting for each sgRNA
    for sgRNA in ['Arid1a_sgRNA1', 'Arid1a_sgRNA2', 'Arid1a_sgRNA3']:
        subset_df = df_with_counts[df_with_counts['GEMorganoid'] == sgRNA]
        
        plt.figure(figsize=(10, 6))
        scatter = sns.scatterplot(data=subset_df, x='mean', y='image_frame_y', hue='C_to_SLC_binned', palette='viridis', legend='full')
        plt.xlabel('Mean')
        plt.ylabel('Frame Number')
        plt.title(sgRNA)
        
        # Modify legend title
        handles, labels = scatter.get_legend_handles_labels()
        scatter.legend(handles=handles, labels=labels, title='C_to_SLC')

        plt.show()


# 10. Appendix - Library versions

## Library versions

In [ ]:
import pandas, numpy, matplotlib, seaborn, scipy

print ("Pandas : " + pandas.__version__)
print ("Numpy : " + numpy.__version__)
print ("Matplotlib : " + matplotlib.__version__)
print ("Seaborn : " + seaborn.__version__)
print ("Scipy : " + scipy.__version__)


Pandas : 2.0.3
Numpy : 1.24.3
Matplotlib : 3.7.1
Seaborn : 0.12.2
Scipy : 1.11.1